# Notebook 05 — Graph Algorithms Fraud Detection

## Fraud Graph Analytics  
### Algoritmos de Grafo para Detecção de Entidades e Comunidades Suspeitas

Este notebook aplica algoritmos de grafo sobre as entidades e relacionamentos preparados no **Notebook 04 — Graph Modeling Neo4j**.

O objetivo é identificar padrões estruturais relevantes para prevenção a fraudes, como:

- contas altamente conectadas;
- dispositivos compartilhados;
- beneficiários concentradores;
- IPs usados por múltiplas contas;
- componentes conectados;
- comunidades suspeitas;
- possíveis contas ponte;
- entidades com alta relevância estrutural.

A análise será conduzida localmente com **NetworkX**, criando uma camada de features de grafo que poderá ser usada no próximo notebook para composição do score de risco explicável.

## 1. Objetivo da Célula

### Objetivo

Configurar o ambiente inicial, carregar os artefatos analíticos dos notebooks anteriores e preparar a base para construção do grafo de investigação.

### Ações realizadas

- Importação das bibliotecas principais.
- Definição dos diretórios do projeto.
- Carregamento da base de transações com score de regras.
- Carregamento dos alertas antifraude.
- Preparação dos diretórios de saída.
- Definição de parâmetros de execução dos algoritmos de grafo.

### Justificativa técnica

Algoritmos de grafo permitem capturar relações indiretas que não aparecem claramente em tabelas tradicionais. Em prevenção a fraudes, esse tipo de análise é útil para identificar redes coordenadas, contas intermediárias, beneficiários concentradores e dispositivos compartilhados.

### Resultados esperados

Ambiente preparado para criar a projeção de grafo, calcular métricas estruturais e exportar features para uso nos próximos notebooks.

In [1]:
from pathlib import Path

import networkx as nx  # noqa: F401
import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", "{:,.4f}".format)

In [3]:
SEED = 42
rng = np.random.default_rng(SEED)

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
GOLD_DIR = DATA_DIR / "03-gold"

DOCS_DIR = PROJECT_ROOT / "docs"
CYPHER_DIR = PROJECT_ROOT / "cypher"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
CYPHER_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Synthetic:    {SYNTHETIC_DIR}")
print(f"Gold:         {GOLD_DIR}")
print(f"Docs:         {DOCS_DIR}")
print(f"Reports:      {REPORTS_DIR}")

Project root: d:\_DS-Projects\Data-Science\fraud-graph-analytics
Synthetic:    d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\synthetic
Gold:         d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold
Docs:         d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs
Reports:      d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports


## 2. Carregamento dos Dados Analíticos

Nesta etapa serão carregados os artefatos gerados no Notebook 03.

A base principal será:

`transactions_with_rule_scores.parquet`

Ela contém transações enriquecidas com:

- regras acionadas;
- score de regras;
- faixa de risco;
- alerta gerado;
- informações de conta, dispositivo, beneficiário e IP.

Também será carregada a base:

`antifraud_alerts.parquet`

Ela representa o subconjunto de transações priorizadas pelo motor de regras.

In [5]:
rules_scored_path = GOLD_DIR / "transactions_with_rule_scores.parquet"
alerts_path = GOLD_DIR / "antifraud_alerts.parquet"

if not rules_scored_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {rules_scored_path}. "
        "Execute o Notebook 03 antes de continuar."
    )

if not alerts_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {alerts_path}. "
        "Execute o Notebook 03 antes de continuar."
    )

rules_scored = pd.read_parquet(rules_scored_path)
alerts = pd.read_parquet(alerts_path)

rules_scored["data_hora"] = pd.to_datetime(rules_scored["data_hora"])
alerts["data_hora"] = pd.to_datetime(alerts["data_hora"])

print("Dados carregados com sucesso.")
print(f"Transações com score: {rules_scored.shape}")
print(f"Alertas antifraude:   {alerts.shape}")

Dados carregados com sucesso.
Transações com score: (80000, 82)
Alertas antifraude:   (79997, 33)


In [6]:
base_summary = pd.DataFrame(
    [
        {
            "dataset": "rules_scored",
            "linhas": len(rules_scored),
            "colunas": rules_scored.shape[1],
            "memoria_mb": round(rules_scored.memory_usage(deep=True).sum() / 1024**2, 2),
        },
        {
            "dataset": "alerts",
            "linhas": len(alerts),
            "colunas": alerts.shape[1],
            "memoria_mb": round(alerts.memory_usage(deep=True).sum() / 1024**2, 2),
        },
    ]
)

base_summary

,dataset,linhas,colunas,memoria_mb
0,rules_scored,80000,82,174.5600
1,alerts,79997,33,83.0800


## 3. Projeção Analítica do Grafo

Para esta etapa, será criada uma projeção de grafo com quatro tipos principais de nós:

- `Conta`;
- `Dispositivo`;
- `Beneficiario`;
- `IP`.

As transações serão usadas para gerar relacionamentos agregados:

- `Conta — Dispositivo`;
- `Conta — Beneficiario`;
- `Conta — IP`.

Essa projeção é adequada para investigar conexões indiretas, compartilhamentos e agrupamentos suspeitos.

In [7]:
def create_agg_edges(
    df: pd.DataFrame,
    source_col: str,
    target_col: str,
    source_prefix: str,
    target_prefix: str,
    relation_type: str,
) -> pd.DataFrame:
    edges = (
        df
        .dropna(subset=[source_col, target_col])
        .groupby([source_col, target_col])
        .agg(
            qtd_transacoes=("transacao_id", "count"),
            valor_total=("valor", "sum"),
            valor_medio=("valor", "mean"),
            score_medio=("rule_score", "mean"),
            score_max=("rule_score", "max"),
            qtd_alertas=("alerta_gerado", "sum"),
            taxa_fraude_sintetica=("is_fraud", "mean"),
            cenarios_distintos=("fraud_scenario", "nunique"),
        )
        .reset_index()
    )

    edges["source"] = source_prefix + ":" + edges[source_col].astype(str)
    edges["target"] = target_prefix + ":" + edges[target_col].astype(str)
    edges["relation_type"] = relation_type

    edges["weight"] = (
        1
        + edges["qtd_transacoes"].clip(upper=20)
        + edges["qtd_alertas"].clip(upper=20) * 2
        + edges["score_medio"].fillna(0) / 10
    )

    return edges


edges_account_device = create_agg_edges(
    rules_scored,
    source_col="conta_origem_id",
    target_col="device_id",
    source_prefix="Conta",
    target_prefix="Dispositivo",
    relation_type="USOU_DISPOSITIVO",
)

edges_account_beneficiary = create_agg_edges(
    rules_scored,
    source_col="conta_origem_id",
    target_col="beneficiario_id",
    source_prefix="Conta",
    target_prefix="Beneficiario",
    relation_type="ENVIOU_PARA_BENEFICIARIO",
)

edges_account_ip = create_agg_edges(
    rules_scored,
    source_col="conta_origem_id",
    target_col="ip_id",
    source_prefix="Conta",
    target_prefix="IP",
    relation_type="USOU_IP",
)

graph_edges = pd.concat(
    [
        edges_account_device,
        edges_account_beneficiary,
        edges_account_ip,
    ],
    ignore_index=True,
)

graph_edges.head()

,conta_origem_id,device_id,qtd_transacoes,valor_total,valor_medio,score_medio,score_max,qtd_alertas,taxa_fraude_sintetica,cenarios_distintos,source,target,relation_type,weight,beneficiario_id,ip_id
0,CTA_000001,DEV_000478,1,631.7400,631.7400,32.0000,32,1,0.0000,1,Conta:CTA_000001,Dispositivo:DEV_000478,USOU_DISPOSITIVO,7.2000,NaN,NaN
1,CTA_000001,DEV_000789,1,104.0900,104.0900,32.0000,32,1,0.0000,1,Conta:CTA_000001,Dispositivo:DEV_000789,USOU_DISPOSITIVO,7.2000,NaN,NaN
2,CTA_000001,DEV_001310,1,232.7600,232.7600,32.0000,32,1,0.0000,1,Conta:CTA_000001,Dispositivo:DEV_001310,USOU_DISPOSITIVO,7.2000,NaN,NaN
3,CTA_000001,DEV_001330,1,385.0600,385.0600,52.0000,52,1,0.0000,1,Conta:CTA_000001,Dispositivo:DEV_001330,USOU_DISPOSITIVO,9.2000,NaN,NaN
4,CTA_000001,DEV_001375,1,184.2000,184.2000,52.0000,52,1,0.0000,1,Conta:CTA_000001,Dispositivo:DEV_001375,USOU_DISPOSITIVO,9.2000,NaN,NaN


In [8]:
edge_summary = (
    graph_edges
    .groupby("relation_type")
    .agg(
        qtd_edges=("source", "count"),
        peso_medio=("weight", "mean"),
        qtd_alertas=("qtd_alertas", "sum"),
        score_medio=("score_medio", "mean"),
        valor_total=("valor_total", "sum"),
    )
    .reset_index()
    .sort_values("qtd_edges", ascending=False)
)

edge_summary

,relation_type,qtd_edges,peso_medio,qtd_alertas,score_medio,valor_total
0,ENVIOU_PARA_BENEFICIARIO,79411,8.4597,79997,44.3757,"81,005,749.9300"
1,USOU_DISPOSITIVO,79329,8.4545,79997,44.2925,"81,005,749.9300"
2,USOU_IP,79326,8.4563,79997,44.3093,"81,005,749.9300"


## 4. Construção do Grafo com NetworkX

Nesta etapa será criado um grafo não direcionado.

A escolha por grafo não direcionado nesta projeção inicial facilita a análise de conectividade, comunidades e centralidade estrutural entre contas, dispositivos, beneficiários e IPs.

Nos notebooks e scripts Cypher, a modelagem orientada a eventos continua preservada.

In [9]:
G = nx.Graph()

for _, row in graph_edges.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        relation_type=row["relation_type"],
        weight=float(row["weight"]),
        qtd_transacoes=int(row["qtd_transacoes"]),
        valor_total=float(row["valor_total"]),
        score_medio=float(row["score_medio"]),
        score_max=float(row["score_max"]),
        qtd_alertas=int(row["qtd_alertas"]),
        taxa_fraude_sintetica=float(row["taxa_fraude_sintetica"]),
    )

for node in G.nodes:
    if ":" in node:
        node_type, raw_id = node.split(":", 1)
    else:
        node_type, raw_id = "Unknown", node

    G.nodes[node]["node_type"] = node_type
    G.nodes[node]["entity_id"] = raw_id

graph_basic_stats = pd.DataFrame(
    [
        {"metrica": "qtd_nos", "valor": G.number_of_nodes()},
        {"metrica": "qtd_arestas", "valor": G.number_of_edges()},
        {"metrica": "densidade", "valor": nx.density(G)},
        {"metrica": "qtd_componentes_conectados", "valor": nx.number_connected_components(G)},
    ]
)

graph_basic_stats

,metrica,valor
0,qtd_nos,"17,000.0000"
1,qtd_arestas,"238,066.0000"
2,densidade,0.0016
3,qtd_componentes_conectados,1.0000


In [10]:
node_type_distribution = (
    pd.Series(nx.get_node_attributes(G, "node_type"))
    .value_counts()
    .rename_axis("node_type")
    .reset_index(name="qtd_nos")
)

node_type_distribution

,node_type,qtd_nos
0,Conta,6000
1,Dispositivo,4500
2,Beneficiario,3500
3,IP,3000


## 5. Degree Centrality e Weighted Degree

O primeiro conjunto de métricas mede conectividade.

### Degree

Quantidade de conexões de um nó.

### Weighted Degree

Soma dos pesos das conexões.

Em fraude, essas métricas ajudam a identificar:

- dispositivos usados por muitas contas;
- beneficiários que recebem de muitas contas;
- IPs compartilhados por muitos acessos;
- contas conectadas a muitas entidades.

In [11]:
degree_dict = dict(G.degree())
weighted_degree_dict = dict(G.degree(weight="weight"))
degree_centrality_dict = nx.degree_centrality(G)

graph_features = pd.DataFrame(
    {
        "node": list(G.nodes()),
        "node_type": [G.nodes[n]["node_type"] for n in G.nodes()],
        "entity_id": [G.nodes[n]["entity_id"] for n in G.nodes()],
        "degree": [degree_dict[n] for n in G.nodes()],
        "weighted_degree": [weighted_degree_dict[n] for n in G.nodes()],
        "degree_centrality": [degree_centrality_dict[n] for n in G.nodes()],
    }
)

graph_features.sort_values("degree", ascending=False).head(20)

,node,node_type,entity_id,degree,weighted_degree,degree_centrality
7003,Conta:CTA_002509,Conta,CTA_002509,254,"3,120.8000",0.0149
4544,Conta:CTA_000664,Conta,CTA_000664,252,"2,755.3000",0.0148
6902,Conta:CTA_002408,Conta,CTA_002408,241,"2,340.2500",0.0142
9834,Conta:CTA_005335,Conta,CTA_005335,240,"2,502.6000",0.0141
5490,Conta:CTA_001134,Conta,CTA_001134,236,"2,847.7500",0.0139
9859,Conta:CTA_005360,Conta,CTA_005360,233,"2,586.9000",0.0137
6888,Conta:CTA_002395,Conta,CTA_002395,231,"2,421.0500",0.0136
6461,Conta:CTA_001976,Conta,CTA_001976,227,"2,506.8500",0.0134
1937,Conta:CTA_000162,Conta,CTA_000162,223,"2,278.8000",0.0131
5047,Conta:CTA_000875,Conta,CTA_000875,222,"2,406.3000",0.0131


## 6. PageRank

O PageRank ajuda a identificar entidades estruturalmente relevantes, considerando não apenas o número de conexões, mas também a importância dos vizinhos.

Em um grafo antifraude, PageRank pode destacar:

- beneficiários centrais;
- dispositivos críticos;
- IPs recorrentes;
- contas inseridas em regiões importantes da rede.

In [12]:
pagerank_dict = nx.pagerank(G, weight="weight", alpha=0.85, max_iter=100)

graph_features["pagerank"] = graph_features["node"].map(pagerank_dict)

graph_features.sort_values("pagerank", ascending=False).head(20)

,node,node_type,entity_id,degree,weighted_degree,degree_centrality,pagerank
7003,Conta:CTA_002509,Conta,CTA_002509,254,"3,120.8000",0.0149,0.0007
5490,Conta:CTA_001134,Conta,CTA_001134,236,"2,847.7500",0.0139,0.0006
4544,Conta:CTA_000664,Conta,CTA_000664,252,"2,755.3000",0.0148,0.0006
9859,Conta:CTA_005360,Conta,CTA_005360,233,"2,586.9000",0.0137,0.0006
9834,Conta:CTA_005335,Conta,CTA_005335,240,"2,502.6000",0.0141,0.0005
6461,Conta:CTA_001976,Conta,CTA_001976,227,"2,506.8500",0.0134,0.0005
5047,Conta:CTA_000875,Conta,CTA_000875,222,"2,406.3000",0.0131,0.0005
6888,Conta:CTA_002395,Conta,CTA_002395,231,"2,421.0500",0.0136,0.0005
6902,Conta:CTA_002408,Conta,CTA_002408,241,"2,340.2500",0.0142,0.0005
1937,Conta:CTA_000162,Conta,CTA_000162,223,"2,278.8000",0.0131,0.0005


## 7. Componentes Conectados

Componentes conectados representam grupos de entidades que possuem algum caminho entre si.

Em prevenção a fraudes, componentes grandes ou com muitos alertas podem indicar regiões da rede com maior complexidade investigativa.

In [13]:
component_records = []

for component_id, nodes in enumerate(nx.connected_components(G), start=1):
    for node in nodes:
        component_records.append(
            {
                "node": node,
                "component_id": component_id,
                "component_size": len(nodes),
            }
        )

components_df = pd.DataFrame(component_records)

graph_features = graph_features.merge(components_df, on="node", how="left")

component_summary = (
    graph_features
    .groupby("component_id")
    .agg(
        component_size=("node", "count"),
        qtd_contas=("node_type", lambda x: (x == "Conta").sum()),
        qtd_dispositivos=("node_type", lambda x: (x == "Dispositivo").sum()),
        qtd_beneficiarios=("node_type", lambda x: (x == "Beneficiario").sum()),
        qtd_ips=("node_type", lambda x: (x == "IP").sum()),
        degree_medio=("degree", "mean"),
        pagerank_total=("pagerank", "sum"),
    )
    .reset_index()
    .sort_values("component_size", ascending=False)
)

component_summary.head(20)

,component_id,component_size,qtd_contas,qtd_dispositivos,qtd_beneficiarios,qtd_ips,degree_medio,pagerank_total
0,1,17000,6000,4500,3500,3000,28.0078,1.0000


## 8. Detecção de Comunidades

Nesta etapa será aplicada detecção de comunidades.

A opção preferencial é Louvain, quando disponível na versão do NetworkX instalada. Caso não esteja disponível, será usado `greedy_modularity_communities` como alternativa.

Comunidades ajudam a identificar grupos densamente conectados que podem representar:

- operação coordenada;
- dispositivos e contas compartilhados;
- beneficiários recorrentes;
- grupos com maior concentração de alertas.

In [14]:
def detect_communities(graph: nx.Graph, seed: int = 42) -> list[set]:
    try:
        communities = nx.community.louvain_communities(
            graph,
            weight="weight",
            seed=seed,
        )
        method = "louvain_communities"
    except Exception:
        communities = list(
            nx.community.greedy_modularity_communities(
                graph,
                weight="weight",
            )
        )
        method = "greedy_modularity_communities"

    return communities, method


communities, community_method = detect_communities(G, seed=SEED)

community_records = []

for community_id, nodes in enumerate(communities, start=1):
    for node in nodes:
        community_records.append(
            {
                "node": node,
                "community_id": community_id,
                "community_size": len(nodes),
                "community_method": community_method,
            }
        )

communities_df = pd.DataFrame(community_records)

graph_features = graph_features.merge(
    communities_df,
    on="node",
    how="left",
)

print(f"Método de comunidade utilizado: {community_method}")
print(f"Quantidade de comunidades: {len(communities)}")

graph_features.head()

Método de comunidade utilizado: louvain_communities
Quantidade de comunidades: 21


,node,node_type,entity_id,degree,weighted_degree,degree_centrality,pagerank,component_id,component_size,community_id,community_size,community_method
0,Conta:CTA_000001,Conta,CTA_000001,42,343.8000,0.0025,0.0001,1,17000,2,1571,louvain_communities
1,Dispositivo:DEV_000478,Dispositivo,DEV_000478,18,148.6000,0.0011,0.0000,1,17000,16,1914,louvain_communities
2,Dispositivo:DEV_000789,Dispositivo,DEV_000789,15,120.8000,0.0009,0.0000,1,17000,2,1571,louvain_communities
3,Dispositivo:DEV_001310,Dispositivo,DEV_001310,24,200.7000,0.0014,0.0000,1,17000,16,1914,louvain_communities
4,Dispositivo:DEV_001330,Dispositivo,DEV_001330,15,130.7000,0.0009,0.0000,1,17000,14,915,louvain_communities


## 9. Betweenness Centrality Aproximado

Betweenness Centrality mede o quanto um nó aparece em caminhos entre outros nós.

Em fraude, essa métrica pode ajudar a identificar:

- contas ponte;
- entidades intermediárias;
- nós que conectam regiões diferentes da rede.

Como essa métrica pode ser custosa em grafos maiores, será usada uma versão aproximada com amostragem.

In [15]:
n_nodes = G.number_of_nodes()
sample_k = min(300, n_nodes)

betweenness_dict = nx.betweenness_centrality(
    G,
    k=sample_k,
    seed=SEED,
    weight="weight",
    normalized=True,
)

graph_features["betweenness_approx"] = graph_features["node"].map(betweenness_dict)

graph_features.sort_values("betweenness_approx", ascending=False).head(20)

,node,node_type,entity_id,degree,weighted_degree,degree_centrality,pagerank,component_id,component_size,community_id,community_size,community_method,betweenness_approx
7980,Conta:CTA_003481,Conta,CTA_003481,76,802.6667,0.0045,0.0002,1,17000,8,179,louvain_communities,0.0018
5549,Conta:CTA_001179,Conta,CTA_001179,51,416.4000,0.0030,0.0001,1,17000,12,1320,louvain_communities,0.0018
10099,Conta:CTA_005600,Conta,CTA_005600,66,539.4000,0.0039,0.0001,1,17000,21,831,louvain_communities,0.0018
6858,Conta:CTA_002365,Conta,CTA_002365,51,414.0000,0.0030,0.0001,1,17000,6,650,louvain_communities,0.0017
7691,Conta:CTA_003192,Conta,CTA_003192,39,309.0000,0.0023,0.0001,1,17000,20,895,louvain_communities,0.0015
6356,Conta:CTA_001874,Conta,CTA_001874,57,446.4000,0.0034,0.0001,1,17000,14,915,louvain_communities,0.0015
992,Conta:CTA_000074,Conta,CTA_000074,217,"2,212.2500",0.0128,0.0005,1,17000,13,768,louvain_communities,0.0015
7164,Conta:CTA_002669,Conta,CTA_002669,54,439.8000,0.0032,0.0001,1,17000,21,831,louvain_communities,0.0015
5238,Conta:CTA_000979,Conta,CTA_000979,42,314.4000,0.0025,0.0001,1,17000,9,496,louvain_communities,0.0015
9884,Conta:CTA_005385,Conta,CTA_005385,53,427.8000,0.0031,0.0001,1,17000,16,1914,louvain_communities,0.0015


## 10. Enriquecimento com Sinais Antifraude

Agora as métricas de grafo serão combinadas com sinais derivados dos alertas.

Essa etapa permite criar uma visão integrada entre:

- estrutura da rede;
- motor de regras;
- labels sintéticos;
- score de risco;
- comportamento transacional.

In [16]:
def aggregate_entity_signals(
    df: pd.DataFrame,
    entity_col: str,
    node_prefix: str,
) -> pd.DataFrame:
    agg = (
        df
        .dropna(subset=[entity_col])
        .groupby(entity_col)
        .agg(
            qtd_transacoes=("transacao_id", "count"),
            qtd_alertas=("alerta_gerado", "sum"),
            taxa_alerta=("alerta_gerado", "mean"),
            taxa_fraude_sintetica=("is_fraud", "mean"),
            score_medio=("rule_score", "mean"),
            score_max=("rule_score", "max"),
            valor_total=("valor", "sum"),
            valor_medio=("valor", "mean"),
            qtd_cenarios=("fraud_scenario", "nunique"),
        )
        .reset_index()
        .rename(columns={entity_col: "entity_id"})
    )

    agg["node_type"] = node_prefix
    agg["node"] = node_prefix + ":" + agg["entity_id"].astype(str)

    return agg


account_signals = aggregate_entity_signals(rules_scored, "conta_origem_id", "Conta")
device_signals = aggregate_entity_signals(rules_scored, "device_id", "Dispositivo")
beneficiary_signals = aggregate_entity_signals(rules_scored, "beneficiario_id", "Beneficiario")
ip_signals = aggregate_entity_signals(rules_scored, "ip_id", "IP")

entity_signals = pd.concat(
    [
        account_signals,
        device_signals,
        beneficiary_signals,
        ip_signals,
    ],
    ignore_index=True,
)

graph_features_enriched = graph_features.merge(
    entity_signals.drop(columns=["entity_id", "node_type"]),
    on="node",
    how="left",
)

signal_cols = [
    "qtd_transacoes",
    "qtd_alertas",
    "taxa_alerta",
    "taxa_fraude_sintetica",
    "score_medio",
    "score_max",
    "valor_total",
    "valor_medio",
    "qtd_cenarios",
]

graph_features_enriched[signal_cols] = graph_features_enriched[signal_cols].fillna(0)

graph_features_enriched.head()

,node,node_type,entity_id,degree,weighted_degree,degree_centrality,pagerank,component_id,component_size,community_id,community_size,community_method,betweenness_approx,qtd_transacoes,qtd_alertas,taxa_alerta,taxa_fraude_sintetica,score_medio,score_max,valor_total,valor_medio,qtd_cenarios
0,Conta:CTA_000001,Conta,CTA_000001,42,343.8000,0.0025,0.0001,1,17000,2,1571,louvain_communities,0.0002,14,14,1.0000,0.0714,41.8571,75,"11,352.5400",810.8957,2
1,Dispositivo:DEV_000478,Dispositivo,DEV_000478,18,148.6000,0.0011,0.0000,1,17000,16,1914,louvain_communities,0.0000,18,18,1.0000,0.0000,42.5556,66,"7,540.8300",418.9350,1
2,Dispositivo:DEV_000789,Dispositivo,DEV_000789,15,120.8000,0.0009,0.0000,1,17000,2,1571,louvain_communities,0.0000,15,15,1.0000,0.0000,40.5333,52,"4,300.9000",286.7267,1
3,Dispositivo:DEV_001310,Dispositivo,DEV_001310,24,200.7000,0.0014,0.0000,1,17000,16,1914,louvain_communities,0.0001,24,24,1.0000,0.0833,43.6250,87,"25,462.1100","1,060.9213",3
4,Dispositivo:DEV_001330,Dispositivo,DEV_001330,15,130.7000,0.0009,0.0000,1,17000,14,915,louvain_communities,0.0000,15,15,1.0000,0.0667,47.1333,97,"7,695.7900",513.0527,2


## 11. Community Risk Score

Nesta etapa será calculado um score de risco por comunidade.

O objetivo é priorizar grupos de entidades, não apenas transações isoladas.

O score considera:

- quantidade de alertas;
- score médio das entidades;
- taxa de fraude sintética;
- conectividade;
- diversidade de tipos de nós;
- tamanho da comunidade.

In [17]:
community_risk_summary = (
    graph_features_enriched
    .groupby("community_id")
    .agg(
        community_size=("node", "count"),
        qtd_contas=("node_type", lambda x: (x == "Conta").sum()),
        qtd_dispositivos=("node_type", lambda x: (x == "Dispositivo").sum()),
        qtd_beneficiarios=("node_type", lambda x: (x == "Beneficiario").sum()),
        qtd_ips=("node_type", lambda x: (x == "IP").sum()),
        qtd_alertas=("qtd_alertas", "sum"),
        score_medio=("score_medio", "mean"),
        score_max=("score_max", "max"),
        taxa_alerta_media=("taxa_alerta", "mean"),
        taxa_fraude_sintetica_media=("taxa_fraude_sintetica", "mean"),
        valor_total=("valor_total", "sum"),
        degree_medio=("degree", "mean"),
        weighted_degree_total=("weighted_degree", "sum"),
        pagerank_total=("pagerank", "sum"),
        betweenness_max=("betweenness_approx", "max"),
    )
    .reset_index()
)

community_risk_summary["node_type_diversity"] = (
    (community_risk_summary[["qtd_contas", "qtd_dispositivos", "qtd_beneficiarios", "qtd_ips"]] > 0)
    .sum(axis=1)
)

community_risk_summary["community_risk_score"] = (
    0.25 * community_risk_summary["score_medio"].fillna(0)
    + 0.20 * (community_risk_summary["qtd_alertas"].rank(pct=True) * 100)
    + 0.20 * (community_risk_summary["weighted_degree_total"].rank(pct=True) * 100)
    + 0.15 * (community_risk_summary["pagerank_total"].rank(pct=True) * 100)
    + 0.10 * (community_risk_summary["betweenness_max"].rank(pct=True) * 100)
    + 0.10 * (community_risk_summary["node_type_diversity"] / 4 * 100)
).round(2)

community_risk_summary = community_risk_summary.sort_values(
    "community_risk_score",
    ascending=False,
)

community_risk_summary.head(20)

,community_id,community_size,qtd_contas,qtd_dispositivos,qtd_beneficiarios,qtd_ips,qtd_alertas,score_medio,score_max,taxa_alerta_media,taxa_fraude_sintetica_media,valor_total,degree_medio,weighted_degree_total,pagerank_total,betweenness_max,node_type_diversity,community_risk_score
15,16,1914,696,496,383,339,35919,43.2959,100,1.0000,0.0555,"29,507,734.3500",28.1594,"451,832.1500",0.1126,0.0015,4,82.0100
1,2,1571,568,405,321,277,29335,43.2747,100,0.9999,0.0619,"26,392,519.7200",27.8568,"367,454.4000",0.0917,0.0014,4,77.0100
6,7,1504,533,398,317,256,27471,42.9968,100,1.0000,0.0591,"25,399,698.0400",27.2533,"342,751.9500",0.0859,0.0014,4,74.8000
11,12,1320,466,356,255,243,24012,42.8207,100,0.9999,0.0553,"21,425,810.7200",27.2545,"299,849.4500",0.0753,0.0018,4,74.7500
3,4,1326,472,351,284,219,24418,42.8885,100,1.0000,0.0584,"21,500,445.1200",27.5173,"304,823.6000",0.0763,0.0014,4,73.5800
13,14,915,327,231,179,178,17059,43.6438,100,1.0000,0.0655,"16,534,161.4700",27.7257,"213,625.6000",0.0533,0.0015,4,70.4300
19,20,895,320,244,175,156,16467,43.2590,100,1.0000,0.0601,"15,575,754.6800",27.5821,"207,175.8500",0.0518,0.0015,4,68.2000
20,21,831,300,220,176,135,15310,43.1955,100,1.0000,0.0610,"14,319,705.4000",27.7160,"193,031.5000",0.0482,0.0018,4,66.5100
12,13,768,248,209,174,137,14712,43.9649,100,1.0000,0.0732,"15,943,099.0300",28.5755,"187,417.2500",0.0464,0.0015,4,62.1800
9,10,755,257,199,171,128,14186,43.1486,100,1.0000,0.0620,"12,945,797.0100",28.0477,"177,768.3000",0.0443,0.0014,4,57.4500


## 12. Entity Graph Risk Score

Agora será calculado um score estrutural por entidade.

Esse score não substitui o score de regras criado no Notebook 03.  
Ele adiciona uma camada de risco relacional baseada em centralidade, conectividade e participação em comunidades suspeitas.

In [18]:
community_score_map = community_risk_summary.set_index("community_id")["community_risk_score"].to_dict()

graph_features_enriched["community_risk_score"] = (
    graph_features_enriched["community_id"].map(community_score_map).fillna(0)
)

for col in [
    "degree",
    "weighted_degree",
    "pagerank",
    "betweenness_approx",
    "qtd_alertas",
    "score_max",
    "community_risk_score",
]:
    rank_col = f"{col}_pct_rank"
    graph_features_enriched[rank_col] = graph_features_enriched[col].rank(pct=True).fillna(0)

graph_features_enriched["entity_graph_risk_score"] = (
    0.20 * graph_features_enriched["degree_pct_rank"] * 100
    + 0.20 * graph_features_enriched["weighted_degree_pct_rank"] * 100
    + 0.15 * graph_features_enriched["pagerank_pct_rank"] * 100
    + 0.10 * graph_features_enriched["betweenness_approx_pct_rank"] * 100
    + 0.15 * graph_features_enriched["qtd_alertas_pct_rank"] * 100
    + 0.10 * graph_features_enriched["score_max_pct_rank"] * 100
    + 0.10 * graph_features_enriched["community_risk_score_pct_rank"] * 100
).round(2)


def assign_entity_risk_band(score: float) -> str:
    if score >= 80:
        return "critico"
    if score >= 60:
        return "alto"
    if score >= 35:
        return "medio"
    return "baixo"


graph_features_enriched["entity_graph_risk_band"] = graph_features_enriched[
    "entity_graph_risk_score"
].apply(assign_entity_risk_band)

graph_features_enriched.sort_values("entity_graph_risk_score", ascending=False).head(30)

,node,node_type,entity_id,degree,weighted_degree,degree_centrality,pagerank,component_id,component_size,community_id,community_size,community_method,betweenness_approx,qtd_transacoes,qtd_alertas,taxa_alerta,taxa_fraude_sintetica,score_medio,score_max,valor_total,valor_medio,qtd_cenarios,community_risk_score,degree_pct_rank,weighted_degree_pct_rank,pagerank_pct_rank,betweenness_approx_pct_rank,qtd_alertas_pct_rank,score_max_pct_rank,community_risk_score_pct_rank,entity_graph_risk_score,entity_graph_risk_band
7003,Conta:CTA_002509,Conta,CTA_002509,254,"3,120.8000",0.0149,0.0007,1,17000,16,1914,louvain_communities,0.0008,86,86,1.0000,0.8953,82.2791,100,"525,957.0600","6,115.7798",2,82.0100,1.0000,1.0000,1.0000,0.9896,0.9969,0.9354,0.9437,98.6400,critico
1931,Dispositivo:DEV_000856,Dispositivo,DEV_000856,147,"1,398.3000",0.0086,0.0003,1,17000,16,1914,louvain_communities,0.0010,150,150,1.0000,0.8933,54.4733,96,"74,310.8300",495.4055,2,82.0100,0.9990,0.9981,0.9980,0.9943,1.0000,0.8576,0.9437,97.8700,critico
4544,Conta:CTA_000664,Conta,CTA_000664,252,"2,755.3000",0.0148,0.0006,1,17000,2,1571,louvain_communities,0.0007,85,85,1.0000,0.9294,69.0353,100,"590,316.0600","6,944.8948",2,77.0100,0.9999,0.9999,0.9999,0.9847,0.9968,0.9354,0.8412,97.5600,critico
8125,Conta:CTA_003626,Conta,CTA_003626,135,"1,434.0000",0.0079,0.0003,1,17000,2,1571,louvain_communities,0.0005,45,45,1.0000,0.7333,66.2222,100,"60,066.6000","1,334.8133",3,77.0100,0.9986,0.9982,0.9983,0.9638,0.9941,0.9354,0.8412,97.2300,critico
440,Dispositivo:DEV_002736,Dispositivo,DEV_002736,76,731.8000,0.0045,0.0002,1,17000,2,1571,louvain_communities,0.0004,76,76,1.0000,0.8816,56.2895,100,"114,463.0100","1,506.0922",3,77.0100,0.9935,0.9906,0.9922,0.9119,0.9957,0.9354,0.8412,96.3900,critico
790,Dispositivo:DEV_002048,Dispositivo,DEV_002048,91,873.1000,0.0054,0.0002,1,17000,16,1914,louvain_communities,0.0003,95,95,1.0000,0.7474,54.4737,93,"55,695.3100",586.2664,3,82.0100,0.9959,0.9940,0.9951,0.8878,0.9980,0.8370,0.9437,96.3800,critico
11717,Beneficiario:BEN_002999,Beneficiario,BEN_002999,106,"1,082.6000",0.0062,0.0002,1,17000,2,1571,louvain_communities,0.0003,108,108,1.0000,0.7870,61.4537,100,"180,489.2200","1,671.1965",6,77.0100,0.9970,0.9967,0.9969,0.8796,0.9990,0.9354,0.8412,96.3700,critico
10596,Beneficiario:BEN_000283,Beneficiario,BEN_000283,82,821.3000,0.0048,0.0002,1,17000,16,1914,louvain_communities,0.0002,82,82,1.0000,0.7683,60.1585,100,"98,789.1900","1,204.7462",4,82.0100,0.9947,0.9928,0.9945,0.7879,0.9964,0.9354,0.9437,96.2800,critico
5490,Conta:CTA_001134,Conta,CTA_001134,236,"2,847.7500",0.0139,0.0006,1,17000,7,1504,louvain_communities,0.0004,79,79,1.0000,0.8354,80.4557,100,"433,077.0100","5,481.9875",3,74.8000,0.9998,0.9999,0.9999,0.9414,0.9961,0.9354,0.7508,96.2100,critico
5293,Conta:CTA_001014,Conta,CTA_001014,84,763.8000,0.0049,0.0002,1,17000,2,1571,louvain_communities,0.0014,28,28,1.0000,0.1786,50.9286,100,"183,402.5800","6,550.0921",2,77.0100,0.9951,0.9912,0.9942,0.9990,0.8997,0.9354,0.8412,95.8900,critico


## 13. Rankings Investigativos por Tipo de Entidade

Nesta etapa serão gerados rankings separados para:

- contas;
- dispositivos;
- beneficiários;
- IPs.

Essa visão é útil para analistas de fraude, pois cada tipo de entidade exige uma estratégia investigativa diferente.

In [19]:
ranking_cols = [
    "node",
    "node_type",
    "entity_id",
    "degree",
    "weighted_degree",
    "pagerank",
    "betweenness_approx",
    "community_id",
    "community_size",
    "community_risk_score",
    "qtd_transacoes",
    "qtd_alertas",
    "score_medio",
    "score_max",
    "taxa_alerta",
    "taxa_fraude_sintetica",
    "valor_total",
    "entity_graph_risk_score",
    "entity_graph_risk_band",
]

entity_ranking = (
    graph_features_enriched[ranking_cols]
    .sort_values("entity_graph_risk_score", ascending=False)
    .reset_index(drop=True)
)

top_accounts = entity_ranking.query("node_type == 'Conta'").head(20)
top_devices = entity_ranking.query("node_type == 'Dispositivo'").head(20)
top_beneficiaries = entity_ranking.query("node_type == 'Beneficiario'").head(20)
top_ips = entity_ranking.query("node_type == 'IP'").head(20)

display(top_accounts)
display(top_devices)
display(top_beneficiaries)
display(top_ips)

,node,node_type,entity_id,degree,weighted_degree,pagerank,betweenness_approx,community_id,community_size,community_risk_score,qtd_transacoes,qtd_alertas,score_medio,score_max,taxa_alerta,taxa_fraude_sintetica,valor_total,entity_graph_risk_score,entity_graph_risk_band
0,Conta:CTA_002509,Conta,CTA_002509,254,"3,120.8000",0.0007,0.0008,16,1914,82.0100,86,86,82.2791,100,1.0000,0.8953,"525,957.0600",98.6400,critico
2,Conta:CTA_000664,Conta,CTA_000664,252,"2,755.3000",0.0006,0.0007,2,1571,77.0100,85,85,69.0353,100,1.0000,0.9294,"590,316.0600",97.5600,critico
3,Conta:CTA_003626,Conta,CTA_003626,135,"1,434.0000",0.0003,0.0005,2,1571,77.0100,45,45,66.2222,100,1.0000,0.7333,"60,066.6000",97.2300,critico
8,Conta:CTA_001134,Conta,CTA_001134,236,"2,847.7500",0.0006,0.0004,7,1504,74.8000,79,79,80.4557,100,1.0000,0.8354,"433,077.0100",96.2100,critico
9,Conta:CTA_001014,Conta,CTA_001014,84,763.8000,0.0002,0.0014,2,1571,77.0100,28,28,50.9286,100,1.0000,0.1786,"183,402.5800",95.8900,critico
12,Conta:CTA_003281,Conta,CTA_003281,164,"1,875.5000",0.0004,0.0004,12,1320,74.7500,55,55,74.2909,100,1.0000,0.6909,"92,931.9300",95.2000,critico
13,Conta:CTA_000875,Conta,CTA_000875,222,"2,406.3000",0.0005,0.0011,4,1326,73.5800,75,75,67.9600,100,1.0000,0.8400,"385,964.2600",95.1300,critico
19,Conta:CTA_005335,Conta,CTA_005335,240,"2,502.6000",0.0005,0.0012,20,895,68.2000,81,81,63.8765,100,1.0000,0.8025,"457,827.2000",93.9700,critico
22,Conta:CTA_005360,Conta,CTA_005360,233,"2,586.9000",0.0006,0.0007,21,831,66.5100,79,79,70.6203,100,1.0000,0.8354,"409,791.8000",93.3200,critico
23,Conta:CTA_000903,Conta,CTA_000903,72,710.1000,0.0002,0.0008,7,1504,74.8000,24,24,58.6250,100,1.0000,0.3333,"183,110.8600",93.0800,critico


,node,node_type,entity_id,degree,weighted_degree,pagerank,betweenness_approx,community_id,community_size,community_risk_score,qtd_transacoes,qtd_alertas,score_medio,score_max,taxa_alerta,taxa_fraude_sintetica,valor_total,entity_graph_risk_score,entity_graph_risk_band
1,Dispositivo:DEV_000856,Dispositivo,DEV_000856,147,"1,398.3000",0.0003,0.0010,16,1914,82.0100,150,150,54.4733,96,1.0000,0.8933,"74,310.8300",97.8700,critico
4,Dispositivo:DEV_002736,Dispositivo,DEV_002736,76,731.8000,0.0002,0.0004,2,1571,77.0100,76,76,56.2895,100,1.0000,0.8816,"114,463.0100",96.3900,critico
5,Dispositivo:DEV_002048,Dispositivo,DEV_002048,91,873.1000,0.0002,0.0003,16,1914,82.0100,95,95,54.4737,93,1.0000,0.7474,"55,695.3100",96.3800,critico
10,Dispositivo:DEV_000671,Dispositivo,DEV_000671,79,766.1000,0.0002,0.0003,2,1571,77.0100,79,79,56.9747,100,1.0000,0.8987,"56,482.2000",95.7400,critico
11,Dispositivo:DEV_001848,Dispositivo,DEV_001848,63,614.2000,0.0001,0.0002,16,1914,82.0100,65,65,56.3692,100,1.0000,0.7846,"31,253.7900",95.2000,critico
15,Dispositivo:DEV_004007,Dispositivo,DEV_004007,143,"1,344.3000",0.0003,0.0007,4,1326,73.5800,145,145,53.6897,100,1.0000,0.9241,"60,139.8700",94.9800,critico
16,Dispositivo:DEV_000105,Dispositivo,DEV_000105,68,646.2000,0.0001,0.0003,7,1504,74.8000,69,69,54.6087,100,1.0000,0.8261,"32,608.2000",94.7500,critico
18,Dispositivo:DEV_002307,Dispositivo,DEV_002307,83,782.4000,0.0002,0.0002,16,1914,82.0100,83,83,54.2651,88,1.0000,0.8072,"32,110.0300",94.0600,critico
20,Dispositivo:DEV_000102,Dispositivo,DEV_000102,88,839.5000,0.0002,0.0002,2,1571,77.0100,89,89,55.0674,89,1.0000,0.7978,"40,333.0900",93.5000,critico
30,Dispositivo:DEV_002459,Dispositivo,DEV_002459,88,837.5000,0.0002,0.0004,4,1326,73.5800,88,88,55.1705,89,1.0000,0.7727,"41,954.5400",92.5500,critico


,node,node_type,entity_id,degree,weighted_degree,pagerank,betweenness_approx,community_id,community_size,community_risk_score,qtd_transacoes,qtd_alertas,score_medio,score_max,taxa_alerta,taxa_fraude_sintetica,valor_total,entity_graph_risk_score,entity_graph_risk_band
6,Beneficiario:BEN_002999,Beneficiario,BEN_002999,106,"1,082.6000",0.0002,0.0003,2,1571,77.0100,108,108,61.4537,100,1.0000,0.7870,"180,489.2200",96.3700,critico
7,Beneficiario:BEN_000283,Beneficiario,BEN_000283,82,821.3000,0.0002,0.0002,16,1914,82.0100,82,82,60.1585,100,1.0000,0.7683,"98,789.1900",96.2800,critico
14,Beneficiario:BEN_002517,Beneficiario,BEN_002517,74,740.6000,0.0002,0.0002,2,1571,77.0100,75,75,59.6267,100,1.0000,0.7467,"111,427.7100",95.1200,critico
17,Beneficiario:BEN_000903,Beneficiario,BEN_000903,92,939.7000,0.0002,0.0004,4,1326,73.5800,92,92,62.1413,100,1.0000,0.7717,"127,894.1100",94.3200,critico
21,Beneficiario:BEN_002787,Beneficiario,BEN_002787,93,942.5500,0.0002,0.0002,2,1571,77.0100,94,94,61.1277,87,1.0000,0.7553,"152,518.1000",93.4800,critico
28,Beneficiario:BEN_002308,Beneficiario,BEN_002308,89,886.3000,0.0002,0.0001,7,1504,74.8000,90,90,59.1667,100,1.0000,0.7444,"102,859.4300",92.7200,critico
39,Beneficiario:BEN_003123,Beneficiario,BEN_003123,105,"1,053.6000",0.0002,0.0002,20,895,68.2000,108,108,59.6296,100,1.0000,0.7315,"140,199.2600",91.7600,critico
45,Beneficiario:BEN_003357,Beneficiario,BEN_003357,91,909.9000,0.0002,0.0002,4,1326,73.5800,91,91,59.9890,97,1.0000,0.7363,"126,403.4300",91.4200,critico
51,Beneficiario:BEN_000352,Beneficiario,BEN_000352,94,948.6000,0.0002,0.0002,14,915,70.4300,94,94,60.9149,97,1.0000,0.8511,"140,127.4400",90.9500,critico
57,Beneficiario:BEN_000563,Beneficiario,BEN_000563,84,842.7000,0.0002,0.0001,14,915,70.4300,84,84,60.3214,100,1.0000,0.7857,"143,179.5200",90.7700,critico


,node,node_type,entity_id,degree,weighted_degree,pagerank,betweenness_approx,community_id,community_size,community_risk_score,qtd_transacoes,qtd_alertas,score_medio,score_max,taxa_alerta,taxa_fraude_sintetica,valor_total,entity_graph_risk_score,entity_graph_risk_band
89,IP:IP_002625,IP,IP_002625,45,371.8000,0.0001,0.0006,2,1571,77.0100,47,47,41.0426,87,1.0000,0.0638,"23,404.8100",89.4900,critico
108,IP:IP_001627,IP,IP_001627,44,365.8000,0.0001,0.0004,12,1320,74.7500,44,44,43.1364,100,1.0000,0.1364,"19,919.9000",88.9700,critico
119,IP:IP_000902,IP,IP_000902,42,358.3000,0.0001,0.0009,12,1320,74.7500,42,42,45.3095,100,1.0000,0.0952,"41,147.5700",88.6600,critico
136,IP:IP_000962,IP,IP_000962,40,341.8000,0.0001,0.0005,7,1504,74.8000,40,40,45.4500,100,1.0000,0.1500,"26,826.4900",87.9300,critico
162,IP:IP_001997,IP,IP_001997,91,"1,253.1433",0.0002,0.0001,8,179,44.4100,145,145,85.2690,100,1.0000,0.7655,"1,033,160.5500",86.9900,critico
169,IP:IP_000467,IP,IP_000467,40,333.6000,0.0001,0.0005,12,1320,74.7500,40,40,43.4000,100,1.0000,0.1000,"42,231.6100",86.7600,critico
177,IP:IP_002237,IP,IP_002237,41,351.5000,0.0001,0.0004,12,1320,74.7500,41,41,45.7317,97,1.0000,0.0732,"84,089.3000",86.6100,critico
215,IP:IP_002069,IP,IP_002069,41,338.1000,0.0001,0.0006,12,1320,74.7500,41,41,42.4634,89,1.0000,0.0732,"28,443.7400",85.6600,critico
223,IP:IP_001636,IP,IP_001636,39,324.7000,0.0001,0.0014,4,1326,73.5800,39,39,43.2564,100,1.0000,0.1538,"37,404.3000",85.4400,critico
230,IP:IP_000128,IP,IP_000128,39,322.0000,0.0001,0.0005,16,1914,82.0100,39,39,42.5641,75,1.0000,0.0256,"26,286.3900",85.2200,critico


## 14. Análise de Comunidades Suspeitas

Agora serão detalhadas as comunidades com maior risco estrutural.

Essa análise ajuda a responder:

- quais comunidades possuem maior risco?
- quantas contas existem em cada comunidade?
- quais entidades centrais aparecem nelas?
- quais comunidades concentram alertas?
- quais comunidades devem ser priorizadas para investigação?

In [20]:
top_community_ids = community_risk_summary.head(10)["community_id"].tolist()

top_community_entities = (
    entity_ranking
    .loc[entity_ranking["community_id"].isin(top_community_ids)]
    .sort_values(["community_risk_score", "entity_graph_risk_score"], ascending=False)
)

top_community_entities.head(50)

,node,node_type,entity_id,degree,weighted_degree,pagerank,betweenness_approx,community_id,community_size,community_risk_score,qtd_transacoes,qtd_alertas,score_medio,score_max,taxa_alerta,taxa_fraude_sintetica,valor_total,entity_graph_risk_score,entity_graph_risk_band
0,Conta:CTA_002509,Conta,CTA_002509,254,"3,120.8000",0.0007,0.0008,16,1914,82.0100,86,86,82.2791,100,1.0000,0.8953,"525,957.0600",98.6400,critico
1,Dispositivo:DEV_000856,Dispositivo,DEV_000856,147,"1,398.3000",0.0003,0.0010,16,1914,82.0100,150,150,54.4733,96,1.0000,0.8933,"74,310.8300",97.8700,critico
5,Dispositivo:DEV_002048,Dispositivo,DEV_002048,91,873.1000,0.0002,0.0003,16,1914,82.0100,95,95,54.4737,93,1.0000,0.7474,"55,695.3100",96.3800,critico
7,Beneficiario:BEN_000283,Beneficiario,BEN_000283,82,821.3000,0.0002,0.0002,16,1914,82.0100,82,82,60.1585,100,1.0000,0.7683,"98,789.1900",96.2800,critico
11,Dispositivo:DEV_001848,Dispositivo,DEV_001848,63,614.2000,0.0001,0.0002,16,1914,82.0100,65,65,56.3692,100,1.0000,0.7846,"31,253.7900",95.2000,critico
18,Dispositivo:DEV_002307,Dispositivo,DEV_002307,83,782.4000,0.0002,0.0002,16,1914,82.0100,83,83,54.2651,88,1.0000,0.8072,"32,110.0300",94.0600,critico
34,Dispositivo:DEV_000555,Dispositivo,DEV_000555,75,713.0000,0.0002,0.0001,16,1914,82.0100,75,75,55.0667,86,1.0000,0.7333,"24,400.9300",92.1500,critico
44,Conta:CTA_003593,Conta,CTA_003593,66,608.4000,0.0001,0.0002,16,1914,82.0100,22,22,52.1818,100,1.0000,0.2273,"93,545.0600",91.5000,critico
50,Conta:CTA_003563,Conta,CTA_003563,63,551.7000,0.0001,0.0003,16,1914,82.0100,21,21,47.5714,94,1.0000,0.0000,"13,889.7900",90.9600,critico
65,Conta:CTA_000384,Conta,CTA_000384,60,582.0000,0.0001,0.0002,16,1914,82.0100,20,20,57.0000,100,1.0000,0.2000,"130,952.3100",90.4400,critico


In [21]:
community_top_entities = []

for community_id in top_community_ids:
    temp = (
        entity_ranking
        .loc[entity_ranking["community_id"] == community_id]
        .sort_values("entity_graph_risk_score", ascending=False)
        .head(10)
        .copy()
    )
    temp["community_id"] = community_id
    community_top_entities.append(temp)

community_top_entities_df = pd.concat(community_top_entities, ignore_index=True)

community_top_entities_df[
    [
        "community_id",
        "node_type",
        "entity_id",
        "degree",
        "pagerank",
        "qtd_alertas",
        "score_max",
        "entity_graph_risk_score",
        "entity_graph_risk_band",
    ]
].head(50)

,community_id,node_type,entity_id,degree,pagerank,qtd_alertas,score_max,entity_graph_risk_score,entity_graph_risk_band
0,16,Conta,CTA_002509,254,0.0007,86,100,98.6400,critico
1,16,Dispositivo,DEV_000856,147,0.0003,150,96,97.8700,critico
2,16,Dispositivo,DEV_002048,91,0.0002,95,93,96.3800,critico
3,16,Beneficiario,BEN_000283,82,0.0002,82,100,96.2800,critico
4,16,Dispositivo,DEV_001848,63,0.0001,65,100,95.2000,critico
5,16,Dispositivo,DEV_002307,83,0.0002,83,88,94.0600,critico
6,16,Dispositivo,DEV_000555,75,0.0002,75,86,92.1500,critico
7,16,Conta,CTA_003593,66,0.0001,22,100,91.5000,critico
8,16,Conta,CTA_003563,63,0.0001,21,94,90.9600,critico
9,16,Conta,CTA_000384,60,0.0001,20,100,90.4400,critico


## 15. Interpretação Executiva dos Algoritmos de Grafo

Nesta etapa serão consolidadas interpretações orientadas a negócio.

O objetivo é traduzir as métricas de grafo para uma linguagem útil em prevenção a fraudes.

In [22]:
graph_algorithm_findings = pd.DataFrame(
    [
        {
            "algoritmo": "Degree / Weighted Degree",
            "achado": "Entidades com muitas conexões ou conexões de alto peso foram destacadas.",
            "interpretacao_antifraude": "Pode indicar beneficiários concentradores, dispositivos compartilhados ou IPs usados por múltiplas contas.",
            "uso_no_proximo_notebook": "Compor variáveis estruturais no score de risco explicável.",
        },
        {
            "algoritmo": "PageRank",
            "achado": "Entidades estruturalmente relevantes foram identificadas considerando a importância dos vizinhos.",
            "interpretacao_antifraude": "Ajuda a priorizar nós que ocupam regiões importantes da rede, mesmo quando não têm o maior grau bruto.",
            "uso_no_proximo_notebook": "Adicionar camada de relevância relacional ao score final.",
        },
        {
            "algoritmo": "Connected Components",
            "achado": "O grafo foi segmentado em grupos conectados.",
            "interpretacao_antifraude": "Componentes maiores podem representar regiões de maior complexidade investigativa.",
            "uso_no_proximo_notebook": "Usar tamanho e composição do componente como sinal contextual.",
        },
        {
            "algoritmo": "Community Detection",
            "achado": "Comunidades foram detectadas a partir da estrutura de conexões.",
            "interpretacao_antifraude": "Comunidades com muitas contas, dispositivos, beneficiários e alertas podem indicar atuação coordenada.",
            "uso_no_proximo_notebook": "Usar community_risk_score como componente do score final.",
        },
        {
            "algoritmo": "Betweenness aproximado",
            "achado": "Nós com potencial papel de ponte foram identificados.",
            "interpretacao_antifraude": "Contas ou entidades ponte podem conectar grupos distintos e merecer investigação prioritária.",
            "uso_no_proximo_notebook": "Adicionar sinal de intermediação estrutural ao score final.",
        },
    ]
)

graph_algorithm_findings

,algoritmo,achado,interpretacao_antifraude,uso_no_proximo_notebook
0,Degree / Weighted Degree,Entidades com muitas conexões ou conexões de alto peso foram destacadas.,"Pode indicar beneficiários concentradores, dispositivos compartilhados ou IPs usados por múltiplas contas.",Compor variáveis estruturais no score de risco explicável.
1,PageRank,Entidades estruturalmente relevantes foram identificadas considerando a importância dos vizinhos.,"Ajuda a priorizar nós que ocupam regiões importantes da rede, mesmo quando não têm o maior grau bruto.",Adicionar camada de relevância relacional ao score final.
2,Connected Components,O grafo foi segmentado em grupos conectados.,Componentes maiores podem representar regiões de maior complexidade investigativa.,Usar tamanho e composição do componente como sinal contextual.
3,Community Detection,Comunidades foram detectadas a partir da estrutura de conexões.,"Comunidades com muitas contas, dispositivos, beneficiários e alertas podem indicar atuação coordenada.",Usar community_risk_score como componente do score final.
4,Betweenness aproximado,Nós com potencial papel de ponte foram identificados.,Contas ou entidades ponte podem conectar grupos distintos e merecer investigação prioritária.,Adicionar sinal de intermediação estrutural ao score final.


## 16. Exportação dos Artefatos de Grafo

Nesta etapa serão exportados os artefatos gerados pelo notebook:

- features de grafo por entidade;
- ranking de entidades suspeitas;
- resumo de risco por comunidade;
- top entidades por comunidade;
- relatório executivo em Markdown;
- script Cypher complementar para GDS.

In [23]:
graph_features_path = GOLD_DIR / "graph_entity_features.parquet"
entity_ranking_path = GOLD_DIR / "graph_entity_ranking.parquet"
community_risk_path = GOLD_DIR / "community_risk_summary.parquet"
community_top_entities_path = GOLD_DIR / "community_top_entities.parquet"

entity_ranking_csv_path = REPORTS_DIR / "graph_entity_ranking_top.csv"
community_risk_csv_path = REPORTS_DIR / "community_risk_summary_top.csv"
community_top_entities_csv_path = REPORTS_DIR / "community_top_entities_top.csv"

graph_features_enriched.to_parquet(graph_features_path, index=False)
entity_ranking.to_parquet(entity_ranking_path, index=False)
community_risk_summary.to_parquet(community_risk_path, index=False)
community_top_entities_df.to_parquet(community_top_entities_path, index=False)

entity_ranking.head(200).to_csv(entity_ranking_csv_path, index=False, encoding="utf-8")
community_risk_summary.head(100).to_csv(community_risk_csv_path, index=False, encoding="utf-8")
community_top_entities_df.head(200).to_csv(community_top_entities_csv_path, index=False, encoding="utf-8")

print("Artefatos exportados:")
print(f"- {graph_features_path}")
print(f"- {entity_ranking_path}")
print(f"- {community_risk_path}")
print(f"- {community_top_entities_path}")
print(f"- {entity_ranking_csv_path}")
print(f"- {community_risk_csv_path}")
print(f"- {community_top_entities_csv_path}")

Artefatos exportados:
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\graph_entity_features.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\graph_entity_ranking.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\community_risk_summary.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\community_top_entities.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\graph_entity_ranking_top.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\community_risk_summary_top.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\community_top_entities_top.csv


In [24]:
gds_enriched_cypher = """
// Graph Data Science — Consultas complementares para Fraud Graph Analytics
// Este script complementa o Notebook 05.
// A execução depende da instalação do Neo4j Graph Data Science.

// 1. Verificar grafos existentes
CALL gds.graph.list()
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount;

// 2. Remover projeção anterior, caso exista
CALL gds.graph.drop('fraud_graph_bipartite', false)
YIELD graphName;

// 3. Projetar grafo Conta-Dispositivo-Beneficiario-IP
CALL gds.graph.project(
    'fraud_graph_bipartite',
    ['Conta', 'Dispositivo', 'Beneficiario', 'IP'],
    {
        USOU_DISPOSITIVO: {
            type: 'USOU_DISPOSITIVO',
            orientation: 'UNDIRECTED',
            properties: ['qtd_alertas', 'score_medio', 'valor_total']
        },
        ENVIOU_PARA_BENEFICIARIO: {
            type: 'ENVIOU_PARA_BENEFICIARIO',
            orientation: 'UNDIRECTED',
            properties: ['qtd_alertas', 'score_medio', 'valor_total']
        },
        USOU_IP: {
            type: 'USOU_IP',
            orientation: 'UNDIRECTED',
            properties: ['qtd_alertas', 'score_medio', 'valor_total']
        }
    }
);

// 4. Degree Centrality
CALL gds.degree.stream('fraud_graph_bipartite')
YIELD nodeId, score
RETURN
    labels(gds.util.asNode(nodeId)) AS labels,
    coalesce(
        gds.util.asNode(nodeId).conta_id,
        gds.util.asNode(nodeId).device_id,
        gds.util.asNode(nodeId).beneficiario_id,
        gds.util.asNode(nodeId).ip_id
    ) AS entity_id,
    score AS degree_score
ORDER BY degree_score DESC
LIMIT 50;

// 5. PageRank
CALL gds.pageRank.stream('fraud_graph_bipartite')
YIELD nodeId, score
RETURN
    labels(gds.util.asNode(nodeId)) AS labels,
    coalesce(
        gds.util.asNode(nodeId).conta_id,
        gds.util.asNode(nodeId).device_id,
        gds.util.asNode(nodeId).beneficiario_id,
        gds.util.asNode(nodeId).ip_id
    ) AS entity_id,
    score AS pagerank_score
ORDER BY pagerank_score DESC
LIMIT 50;

// 6. Weakly Connected Components
CALL gds.wcc.stream('fraud_graph_bipartite')
YIELD nodeId, componentId
RETURN
    componentId,
    count(*) AS component_size
ORDER BY component_size DESC
LIMIT 25;

// 7. Louvain Community Detection
CALL gds.louvain.stream('fraud_graph_bipartite')
YIELD nodeId, communityId
RETURN
    communityId,
    count(*) AS community_size
ORDER BY community_size DESC
LIMIT 25;

// 8. Amostra de entidades por comunidade
CALL gds.louvain.stream('fraud_graph_bipartite')
YIELD nodeId, communityId
WITH communityId, gds.util.asNode(nodeId) AS n
RETURN
    communityId,
    labels(n) AS labels,
    coalesce(n.conta_id, n.device_id, n.beneficiario_id, n.ip_id) AS entity_id
ORDER BY communityId
LIMIT 100;
""".strip()

gds_enriched_path = CYPHER_DIR / "06_gds_fraud_detection_queries.cypher"
gds_enriched_path.write_text(gds_enriched_cypher, encoding="utf-8")

print(f"Script GDS complementar salvo em: {gds_enriched_path}")

Script GDS complementar salvo em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\cypher\06_gds_fraud_detection_queries.cypher


In [25]:
def safe_markdown_table(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return df.to_csv(index=False)


graph_algorithms_summary = pd.DataFrame(
    [
        {
            "item": "Nós no grafo",
            "valor": G.number_of_nodes(),
            "interpretacao": "Entidades conectadas na projeção analítica.",
        },
        {
            "item": "Arestas no grafo",
            "valor": G.number_of_edges(),
            "interpretacao": "Relações agregadas entre contas, dispositivos, beneficiários e IPs.",
        },
        {
            "item": "Componentes conectados",
            "valor": nx.number_connected_components(G),
            "interpretacao": "Grupos conectados por caminhos na rede.",
        },
        {
            "item": "Comunidades detectadas",
            "valor": len(communities),
            "interpretacao": f"Comunidades detectadas pelo método {community_method}.",
        },
        {
            "item": "Entidades com risco crítico",
            "valor": int((entity_ranking["entity_graph_risk_band"] == "critico").sum()),
            "interpretacao": "Nós priorizados pela camada estrutural de grafo.",
        },
    ]
)

report_lines = [
    "# Graph Algorithms Fraud Detection — Resumo Executivo",
    "",
    "Este documento consolida os principais resultados do Notebook 05.",
    "",
    "## Objetivo",
    "",
    "Aplicar algoritmos de grafo para identificar entidades e comunidades suspeitas em uma base sintética de prevenção a fraudes transacionais.",
    "",
    "## Síntese Geral",
    "",
    safe_markdown_table(graph_algorithms_summary),
    "",
    "## Distribuição por Tipo de Nó",
    "",
    safe_markdown_table(node_type_distribution),
    "",
    "## Relações Projetadas",
    "",
    safe_markdown_table(edge_summary),
    "",
    "## Top Comunidades por Risco",
    "",
    safe_markdown_table(community_risk_summary.head(15)),
    "",
    "## Top Entidades por Risco de Grafo",
    "",
    safe_markdown_table(entity_ranking.head(25)),
    "",
    "## Achados dos Algoritmos",
    "",
    safe_markdown_table(graph_algorithm_findings),
    "",
    "## Valor Analítico",
    "",
    "- Identificação de entidades estruturalmente relevantes.",
    "- Priorização de comunidades suspeitas.",
    "- Complemento ao motor de regras antifraude.",
    "- Criação de features relacionais para score explicável.",
    "- Preparação para uso posterior no Neo4j Graph Data Science.",
    "",
    "## Observação",
    "",
    "Os resultados são derivados de dados sintéticos criados exclusivamente para fins educacionais, analíticos e de portfólio.",
    "",
]

graph_algorithms_report_path = DOCS_DIR / "graph_algorithms_fraud_detection_summary.md"
graph_algorithms_report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Relatório executivo salvo em: {graph_algorithms_report_path}")

Relatório executivo salvo em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\graph_algorithms_fraud_detection_summary.md


## 17. Conclusão Executiva do Notebook 05

Este notebook aplicou algoritmos de grafo sobre a rede sintética do projeto **Fraud Graph Analytics**.

Foram construídas métricas estruturais para identificar:

- entidades com alta conectividade;
- dispositivos compartilhados;
- beneficiários concentradores;
- IPs recorrentes;
- componentes conectados;
- comunidades suspeitas;
- possíveis contas ponte;
- entidades com maior risco relacional.

A análise demonstrou que o motor de regras antifraude pode ser enriquecido com uma camada estrutural de grafo, ampliando a capacidade de investigação para além da transação isolada.

As principais saídas deste notebook foram:

- `graph_entity_features.parquet`;
- `graph_entity_ranking.parquet`;
- `community_risk_summary.parquet`;
- `community_top_entities.parquet`;
- relatório executivo em Markdown;
- script Cypher complementar para Neo4j GDS.

O próximo passo será o:

**Notebook 06 — Fraud Risk Score Explainability**

Nesse notebook, o score de regras criado no Notebook 03 será combinado com as features estruturais geradas neste notebook, criando um score final de risco com explicabilidade transacional e relacional.